# XLKD-01: ExpansionNet v2 → S1 Mapped Probability-Mass Preflight

XLKD-00 đã chứng minh ánh xạ cấu trúc:

- 9.959/10.000 Teacher token ánh xạ được;
- không có mapping collision;
- 1.304.640/1.304.640 token occurrence trong pseudo-caption ánh xạ được;
- sequence-length và position alignment đạt 100%.

XLKD-01 kiểm tra phần còn thiếu: **soft probability mass**. ExpansionNet v2
được chạy teacher-forcing trên một mẫu Train+Restval cố định gồm 1.024 ảnh.
Notebook đo:

- mapped/unmapped mass tại `T=4`;
- top-k retained mass cho `k = 32, 64, 128`;
- finite-logit rate và mask correctness;
- GPU peak memory và thời gian.

Không optimizer, không backward, không huấn luyện Student. Validation và
held-out Test không được đọc. Nếu không có `k` vượt gate đã khóa ở XLKD-00,
nhánh sparse mapped Logit KD phải dừng hoặc đăng ký lại một protocol mới trước
khi chạy tiếp.

## Kaggle inputs bắt buộc

1. XLKD-00 output.
2. Phase 01 Formal Output.
3. Phase 03B ExpansionNet v2 Offline Preflight output.
4. Phase 03E ExpansionNet v2 Replacement Handoff output.
5. ExpansionNet v2 offline Model package.
6. MS COCO 2014 image dataset.

Runtime: GPU T4 hoặc mạnh hơn, Internet Off.


## Cell 1 — Authorization, environment và leakage boundary


In [1]:
import csv, gc, hashlib, json, os, pickle, platform, random, sys, time
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
DATASET_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//xlkd01_probability_mass_preflight",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

def find_expansionnet_manifest():
    matches = []
    for candidate in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get("model_id") == "expansionnet_v2":
            matches.append(candidate.resolve())
    assert len(matches) == 1, (
        "Cần đúng một ExpansionNet v2 offline Model input; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]

if not torch.cuda.is_available():
    raise RuntimeError(
        "XLKD-01 yêu cầu GPU. Chọn GPU T4 hoặc mạnh hơn rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

X00_PATH = exactly_one("XLKD00_EXPANSIONNET_S1_COMPATIBILITY_COMPLETE.json")
AUTH_PATH = exactly_one(
    "XLKD01_GPU_PROBABILITY_MASS_PREFLIGHT_AUTHORIZED.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P3B_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
P3E_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
MODEL_MANIFEST_PATH = find_expansionnet_manifest()
MODEL_ROOT = MODEL_MANIFEST_PATH.parent
X00, AUTH, P1, P3B, P3E = map(
    read_json, [X00_PATH, AUTH_PATH, P1_PATH, P3B_PATH, P3E_PATH]
)
assert X00["status"] == "complete"
assert X00["structural_mapping_gate"] == "passed"
assert X00["xlkd01_authorized"] is True
assert X00["full_training_authorized"] is False
assert AUTH["status"] == "authorized"
assert AUTH["full_training_authorized"] is False
assert AUTH["validation_selection_authorized"] is False
assert float(AUTH["temperature"]) == 4.0
assert int(AUTH["fixed_sample_size"]) == 1024
assert AUTH["top_k_candidates"] == [32, 64, 128]
assert P1["status"] == "ready"
assert P3B["status"] == "ready" and P3B["teacher"] == "ExpansionNet v2"
assert P3E["status"] == "complete"
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
for marker in [X00, AUTH, P3B, P3E]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

write_json(AUDIT / "xlkd01_cell01_environment.json", {
    "schema_version": "xlkd01_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "strict_deterministic": True,
    "sample_split": "Train+Restval only",
    "validation_read": False,
    "training_started": False,
    "optimizer_created": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01 CELL 1 — ENVIRONMENT AND AUTHORIZATION")
print("=" * 78)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Temperature / sample      : 4.0 / 1024")
print("Top-k candidates          : 32 / 64 / 128")
print("Training / optimizer      : False / False")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 1 / 7 code cells")


XLKD-01 CELL 1 — ENVIRONMENT AND AUTHORIZATION
Status                    : PASSED
GPU                       : Tesla T4
Temperature / sample      : 4.0 / 1024
Top-k candidates          : 32 / 64 / 128
Training / optimizer      : False / False
Validation / Test         : NOT READ / SEALED
Progress                  : 1 / 7 code cells


## Cell 2 — Integrity, mapping và locked protocol


In [2]:
STARTED = time.perf_counter()

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(marker_path.parent, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

x00_rows = verify_manifest(X00_PATH, X00)
mapping_ref = AUTH["mapping_contract"]
MAPPING_PATH = resolve_inside(
    X00_PATH.parent, mapping_ref["relative_path"]
)
assert sha256(MAPPING_PATH) == mapping_ref["sha256"]
MAPPING_PAYLOAD = read_json(MAPPING_PATH)
mapping = {
    int(source): int(target)
    for source, target in MAPPING_PAYLOAD["mapping"].items()
}
assert len(mapping) == 9959
assert MAPPING_PAYLOAD["mapping_collisions"] == 0
assert float(MAPPING_PAYLOAD["vocabulary_count_coverage"]) == 0.9959

protocol_ref = AUTH["protocol"]
PROTOCOL_PATH = resolve_inside(
    X00_PATH.parent, protocol_ref["relative_path"]
)
assert sha256(PROTOCOL_PATH) == protocol_ref["sha256"]
PROTOCOL = read_json(PROTOCOL_PATH)
assert PROTOCOL["status"] == "structural_preflight_passed"
assert PROTOCOL["direct_raw_logit_kd_allowed"] is False
assert PROTOCOL["full_training_authorized"] is False
NEXT = PROTOCOL["next_stage"]
assert NEXT["fixed_sample_size"] == 1024
assert NEXT["top_k_candidates"] == [32, 64, 128]
GATE = NEXT["go_gate"]

model_manifest = read_json(MODEL_MANIFEST_PATH)
assert model_manifest["model_id"] == "expansionnet_v2"

def verify_model_record(record):
    target = (MODEL_ROOT / record["relative_path"]).resolve()
    assert target.is_relative_to(MODEL_ROOT.resolve())
    assert target.is_file()
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256(target) == record["sha256"]
    return target

CHECKPOINT_PATH = verify_model_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_model_record(model_manifest["vocabulary"])
assert sha256(CHECKPOINT_PATH) == AUTH["teacher_checkpoint_sha256"]
for record in model_manifest["required_files"]:
    verify_model_record(record)
for record in model_manifest["integration_patches"]:
    target = (MODEL_ROOT / record["file"]).resolve()
    assert target.is_file() and sha256(target) == record["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_tokens = pickle.load(handle)
NATIVE_WORD2IDX = native_tokens["word2idx_dict"]
NATIVE_IDX2WORD = native_tokens["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX[native_tokens["sos_str"]])
NATIVE_EOS = int(NATIVE_WORD2IDX[native_tokens["eos_str"]])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
NATIVE_UNK = int(NATIVE_WORD2IDX["UNK"])
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (
    78, 79, 77, 80
)

PSEUDO_PATH = (
    P3E_PATH.parent
    / "pseudocaptions"
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
).resolve()
PSEUDO_SUMMARY_PATH = (
    P3E_PATH.parent
    / "pseudocaptions"
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
).resolve()
assert PSEUDO_PATH.is_file() and PSEUDO_SUMMARY_PATH.is_file()
PSEUDO_SUMMARY = read_json(PSEUDO_SUMMARY_PATH)
assert PSEUDO_SUMMARY["records"] == 113287
assert sha256(PSEUDO_PATH) == PSEUDO_SUMMARY["jsonl_sha256"]
assert PSEUDO_SUMMARY["held_out_test_overlap"] == 0
assert PSEUDO_SUMMARY["validation_overlap"] == 0

write_json(AUDIT / "xlkd01_cell02_integrity.json", {
    "schema_version": "xlkd01_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd00_manifest_rows": len(x00_rows),
    "mapping_tokens": len(mapping),
    "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "pseudo_caption_rows": 113287,
    "checkpoint_payload_read": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01 CELL 2 — INTEGRITY")
print("=" * 78)
print("XLKD-00 manifest          :", len(x00_rows), "/", len(x00_rows))
print("Mapped Teacher tokens     :", len(mapping))
print("Teacher checkpoint        : VERIFIED")
print("Pseudo-caption corpus     : 113,287 VERIFIED")
print("Checkpoint payload read   : False")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 7 code cells")


XLKD-01 CELL 2 — INTEGRITY
XLKD-00 manifest          : 11 / 11
Mapped Teacher tokens     : 9959
Teacher checkpoint        : VERIFIED
Pseudo-caption corpus     : 113,287 VERIFIED
Checkpoint payload read   : False
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 7 code cells


## Cell 3 — Deterministic Train+Restval sample và image loader


In [3]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

records = []
with PSEUDO_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        image_id = int(row["image_id"])
        score = hashlib.sha256(
            f"{SEED}|{image_id}".encode("utf-8")
        ).hexdigest()
        records.append({
            "score": score,
            "image_id": image_id,
            "teacher_token_ids": [
                int(value) for value in row["teacher_token_ids"]
            ],
            "student_token_ids": [
                int(value) for value in row["student_token_ids"]
            ],
        })
assert len(records) == 113287
records.sort(key=lambda row: (row["score"], row["image_id"]))
SAMPLE_RECORDS = records[:1024]
del records
assert len(SAMPLE_RECORDS) == 1024
assert len({row["image_id"] for row in SAMPLE_RECORDS}) == 1024
for row in SAMPLE_RECORDS:
    assert len(row["teacher_token_ids"]) == len(row["student_token_ids"])
    assert [mapping[value] for value in row["teacher_token_ids"]] == (
        row["student_token_ids"]
    )

IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")
sample_ids = {row["image_id"] for row in SAMPLE_RECORDS}
manifest_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
)
sample_manifest = {}
for row in manifest_table.to_pylist():
    image_id = int(row["image_id"])
    if image_id not in sample_ids:
        continue
    assert row["experiment_split"] == "train"
    sample_manifest[image_id] = row
assert len(sample_manifest) == 1024

def find_image_leaf(name, prefix):
    matches = []
    for path in DATASET_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")
for record in SAMPLE_RECORDS:
    source = sample_manifest[record["image_id"]]
    leaf = (
        TRAIN_LEAF
        if source["source_partition"] == "train2014"
        else VAL_LEAF
    )
    record["image_path"] = (
        leaf / Path(source["image_path"]).name
    ).resolve()
    assert record["image_path"].is_file()

PREPROCESS = transforms.Compose([
    transforms.Resize(
        (384, 384), interpolation=InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class ProbabilityMassDataset(Dataset):
    def __len__(self):
        return len(SAMPLE_RECORDS)

    def __getitem__(self, index):
        record = SAMPLE_RECORDS[index]
        with Image.open(record["image_path"]) as source:
            image = PREPROCESS(source.convert("RGB"))
        return {
            "image_id": record["image_id"],
            "image": image,
            "teacher_token_ids": record["teacher_token_ids"],
        }

def collate(batch):
    maximum = max(len(row["teacher_token_ids"]) for row in batch)
    tokens = torch.full(
        (len(batch), maximum), NATIVE_PAD, dtype=torch.long
    )
    for index, row in enumerate(batch):
        sequence = torch.tensor(
            row["teacher_token_ids"], dtype=torch.long
        )
        tokens[index, :len(sequence)] = sequence
    return {
        "image_ids": [row["image_id"] for row in batch],
        "images": torch.stack([row["image"] for row in batch]),
        "teacher_tokens": tokens,
    }

DATASET = ProbabilityMassDataset()
LOADER = DataLoader(
    DATASET,
    # ExpansionNet v2 contains a Swin-L backbone and a 2.8 GB checkpoint.
    # Batch 1 is the conservative configuration already validated in Phase 03B
    # on a single Kaggle T4.
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate,
)
probe = DATASET[0]
assert probe["image"].shape == (3, 384, 384)
assert torch.isfinite(probe["image"]).all()

sample_registry = RESULT / "xlkd01_fixed_sample.json"
write_json(sample_registry, {
    "schema_version": "xlkd01_fixed_sample_v1",
    "created_at_utc": now(),
    "seed": 42,
    "rule": "sort SHA256('42|image_id'), take first 1024",
    "split": "Train+Restval represented as experiment_split=train",
    "sample_size": 1024,
    "image_ids": [row["image_id"] for row in SAMPLE_RECORDS],
    "validation_overlap": 0,
    "test_overlap": 0,
})
write_json(AUDIT / "xlkd01_cell03_sample.json", {
    "schema_version": "xlkd01_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "sample_size": 1024,
    "unique_images": 1024,
    "sequence_alignment": "1024 / 1024",
    "input_shape": [3, 384, 384],
    "validation_overlap": 0,
    "test_overlap": 0,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01 CELL 3 — FIXED TRAIN+RESTVAL SAMPLE")
print("=" * 78)
print("Sample / unique images    : 1024 / 1024")
print("Selection rule            : SHA256 seed-42")
print("Teacher/Student alignment : 1024 / 1024")
print("Input tensor              : [3,384,384] float32")
print("Validation / Test overlap : 0 / 0")
print("Progress                  : 3 / 7 code cells")


XLKD-01 CELL 3 — FIXED TRAIN+RESTVAL SAMPLE
Sample / unique images    : 1024 / 1024
Selection rule            : SHA256 seed-42
Teacher/Student alignment : 1024 / 1024
Input tensor              : [3,384,384] float32
Validation / Test overlap : 0 / 0
Progress                  : 3 / 7 code cells


## Cell 4 — Strict-load frozen ExpansionNet v2


In [4]:
STARTED = time.perf_counter()
SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384,
    swin_patch_size=4,
    swin_in_chans=3,
    swin_embed_dim=192,
    swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48],
    swin_window_size=12,
    swin_mlp_ratio=4.0,
    swin_qkv_bias=True,
    swin_qk_scale=None,
    swin_drop_rate=0.0,
    swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0,
    swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False,
    swin_patch_norm=True,
    swin_use_checkpoint=False,
    final_swin_dim=1536,
    d_model=512,
    N_enc=3,
    N_dec=3,
    num_heads=8,
    ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16,
    output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD,
    max_seq_len=74,
    drop_args=drop_args,
    rank=device,
)
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
assert "model_state_dict" in checkpoint
incompatible = TEACHER.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
assert not incompatible.missing_keys
assert not incompatible.unexpected_keys
del checkpoint
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
PARAMETERS = sum(parameter.numel() for parameter in TEACHER.parameters())
TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in TEACHER.parameters()
    if parameter.requires_grad
)
assert TRAINABLE_PARAMETERS == 0 and TEACHER.training is False

write_json(AUDIT / "xlkd01_cell04_teacher.json", {
    "schema_version": "xlkd01_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "parameters": PARAMETERS,
    "strict_load": True,
    "missing_keys": 0,
    "unexpected_keys": 0,
    "frozen_teacher": True,
    "optimizer_created": False,
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01 CELL 4 — TEACHER LOAD")
print("=" * 78)
print("Strict checkpoint load    : PASSED")
print("Parameters                :", PARAMETERS)
print("Trainable parameters      : 0")
print("Optimizer / training      : False / False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 7 code cells")


XLKD-01 CELL 4 — TEACHER LOAD
Strict checkpoint load    : PASSED
Parameters                : 233803076
Trainable parameters      : 0
Optimizer / training      : False / False
Test Set status           : SEALED
Progress                  : 4 / 7 code cells


## Cell 5 — Run mapped-mass and top-k measurements


In [5]:
STARTED = time.perf_counter()
TEMPERATURE = 4.0
TOP_K = [32, 64, 128]
mapped_teacher_indices = torch.tensor(
    sorted(mapping), dtype=torch.long, device=device
)
unmapped_teacher_indices = torch.tensor(
    [index for index in range(10000) if index not in mapping],
    dtype=torch.long,
    device=device,
)
assert mapped_teacher_indices.numel() == 9959
assert unmapped_teacher_indices.numel() == 41

mapped_mass_values = []
unmapped_mass_values = []
topk_values = {value: [] for value in TOP_K}
finite_positions = 0
valid_positions = 0
position_mask_correct = 0
processed_images = 0

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
inference_started = time.perf_counter()
with torch.inference_mode():
    for batch_index, batch in enumerate(LOADER):
        images = batch["images"].to(device, non_blocking=True)
        tokens = batch["teacher_tokens"].to(device, non_blocking=True)
        decoder_input = tokens[:, :-1]
        decoder_labels = tokens[:, 1:]
        decoder_num_pads = (
            decoder_input == NATIVE_PAD
        ).sum(dim=1).tolist()
        logits = TEACHER(
            enc_x=images,
            dec_x=decoder_input,
            enc_x_num_pads=[0] * images.shape[0],
            dec_x_num_pads=[int(value) for value in decoder_num_pads],
            apply_log_softmax=False,
            mode="forward",
        )
        assert logits.shape[:2] == decoder_input.shape
        assert logits.shape[-1] == 10000
        valid_mask = decoder_labels != NATIVE_PAD
        expected_valid = sum(
            len(SAMPLE_RECORDS[processed_images + index][
                "teacher_token_ids"
            ]) - 1
            for index in range(images.shape[0])
        )
        actual_valid = int(valid_mask.sum().item())
        if actual_valid == expected_valid:
            position_mask_correct += actual_valid
        valid_logits = logits[valid_mask].float()
        valid_positions += actual_valid
        finite = torch.isfinite(valid_logits).all(dim=-1)
        finite_positions += int(finite.sum().item())
        assert bool(finite.all())
        probabilities = torch.softmax(
            valid_logits / TEMPERATURE, dim=-1
        )
        mapped_probabilities = probabilities.index_select(
            dim=-1, index=mapped_teacher_indices
        )
        mapped_mass = mapped_probabilities.sum(dim=-1)
        unmapped_mass = probabilities.index_select(
            dim=-1, index=unmapped_teacher_indices
        ).sum(dim=-1)
        assert torch.allclose(
            mapped_mass + unmapped_mass,
            torch.ones_like(mapped_mass),
            atol=2e-5,
            rtol=0.0,
        )
        mapped_mass_values.extend(
            mapped_mass.detach().cpu().double().tolist()
        )
        unmapped_mass_values.extend(
            unmapped_mass.detach().cpu().double().tolist()
        )
        for k in TOP_K:
            retained = torch.topk(
                mapped_probabilities, k=k, dim=-1
            ).values.sum(dim=-1)
            topk_values[k].extend(
                retained.detach().cpu().double().tolist()
            )
        processed_images += images.shape[0]
        if processed_images % 128 == 0:
            print(
                f"processed={processed_images}/1024 | "
                f"valid_positions={valid_positions}"
            )
        del (
            images, tokens, decoder_input, decoder_labels, logits,
            valid_logits, probabilities, mapped_probabilities,
            mapped_mass, unmapped_mass,
        )

inference_seconds = time.perf_counter() - inference_started
peak_memory_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
assert processed_images == 1024
assert valid_positions == len(mapped_mass_values)
assert valid_positions == len(unmapped_mass_values)
assert all(len(topk_values[k]) == valid_positions for k in TOP_K)
finite_rate = finite_positions / valid_positions
mask_correctness = position_mask_correct / valid_positions

raw_npz = RESULT / "xlkd01_probability_mass_samples.npz"
np.savez_compressed(
    raw_npz,
    mapped_mass=np.asarray(mapped_mass_values, dtype=np.float32),
    unmapped_mass=np.asarray(unmapped_mass_values, dtype=np.float32),
    top32_mass=np.asarray(topk_values[32], dtype=np.float32),
    top64_mass=np.asarray(topk_values[64], dtype=np.float32),
    top128_mass=np.asarray(topk_values[128], dtype=np.float32),
)
write_json(AUDIT / "xlkd01_cell05_inference.json", {
    "schema_version": "xlkd01_cell05_v1",
    "created_at_utc": now(),
    "status": "passed",
    "images": processed_images,
    "valid_positions": valid_positions,
    "finite_logit_rate": finite_rate,
    "position_mask_correctness": mask_correctness,
    "temperature": TEMPERATURE,
    "top_k_candidates": TOP_K,
    "inference_seconds": inference_seconds,
    "images_per_second": processed_images / inference_seconds,
    "peak_gpu_memory_mib": peak_memory_mib,
    "optimizer_created": False,
    "backward_performed": False,
    "training_started": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01 CELL 5 — GPU INFERENCE")
print("=" * 78)
print("Images / valid positions  :", processed_images, "/", valid_positions)
print("Finite-logit rate         :", f"{finite_rate:.6%}")
print("Mask correctness          :", f"{mask_correctness:.6%}")
print("Inference seconds         :", round(inference_seconds, 2))
print("Images / second           :", round(processed_images / inference_seconds, 4))
print("Peak GPU memory           :", round(peak_memory_mib, 2), "MiB")
print("Optimizer / backward      : False / False")
print("Progress                  : 5 / 7 code cells")


processed=128/1024 | valid_positions=1344
processed=256/1024 | valid_positions=2698
processed=384/1024 | valid_positions=4060
processed=512/1024 | valid_positions=5405
processed=640/1024 | valid_positions=6764
processed=768/1024 | valid_positions=8097
processed=896/1024 | valid_positions=9468
processed=1024/1024 | valid_positions=10800
XLKD-01 CELL 5 — GPU INFERENCE
Images / valid positions  : 1024 / 10800
Finite-logit rate         : 100.000000%
Mask correctness          : 100.000000%
Inference seconds         : 103.92
Images / second           : 9.8533
Peak GPU memory           : 1042.04 MiB
Optimizer / backward      : False / False
Progress                  : 5 / 7 code cells


## Cell 6 — Summarize gates và select/stop sparse top-k


In [6]:
STARTED = time.perf_counter()

def stats(values):
    array = np.asarray(values, dtype=np.float64)
    return {
        "n": int(array.size),
        "mean": float(array.mean()),
        "p05": float(np.quantile(array, 0.05)),
        "p50": float(np.quantile(array, 0.50)),
        "p95": float(np.quantile(array, 0.95)),
        "minimum": float(array.min()),
        "maximum": float(array.max()),
    }

mapped_stats = stats(mapped_mass_values)
unmapped_stats = stats(unmapped_mass_values)
topk_stats = {str(k): stats(topk_values[k]) for k in TOP_K}
mapped_gate = (
    mapped_stats["mean"]
    >= float(GATE["mean_mapped_probability_mass_min"])
    and mapped_stats["p05"]
    >= float(GATE["p05_mapped_probability_mass_min"])
)
finite_gate = finite_rate == float(GATE["finite_logit_rate"])
mask_gate = (
    mask_correctness == float(GATE["position_mask_correctness"])
)
passing_topk = [
    k for k in TOP_K
    if topk_stats[str(k)]["mean"] >= 0.98
    and topk_stats[str(k)]["p05"] >= 0.95
]
selected_top_k = min(passing_topk) if passing_topk else None
topk_gate = selected_top_k is not None
overall_gate = mapped_gate and finite_gate and mask_gate and topk_gate
decision = (
    "authorize_xlkd02_loss_preflight"
    if overall_gate
    else "stop_sparse_topk_branch"
)

summary = {
    "schema_version": "xlkd01_probability_mass_summary_v1",
    "created_at_utc": now(),
    "status": "passed",
    "temperature": TEMPERATURE,
    "sample_size": 1024,
    "valid_positions": valid_positions,
    "mapped_probability_mass": mapped_stats,
    "unmapped_probability_mass": unmapped_stats,
    "top_k_retained_mass": topk_stats,
    "finite_logit_rate": finite_rate,
    "position_mask_correctness": mask_correctness,
    "gates": {
        "mapped_mass": mapped_gate,
        "finite_logits": finite_gate,
        "position_mask": mask_gate,
        "sparse_top_k": topk_gate,
        "overall": overall_gate,
    },
    "selected_top_k": selected_top_k,
    "decision": decision,
    "scientific_interpretation": (
        "Vocabulary mapping feasibility and sparse approximation feasibility "
        "are separate questions. Failure of top-k retention does not make the "
        "vocabulary map incorrect; it means the preregistered sparse corpus "
        "would discard too much softened Teacher probability mass."
    ),
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
SUMMARY_PATH = RESULT / "xlkd01_probability_mass_summary.json"
write_json(SUMMARY_PATH, summary)
summary_csv = RESULT / "xlkd01_probability_mass_summary.csv"
summary_rows = [
    {"distribution": "mapped", **mapped_stats},
    {"distribution": "unmapped", **unmapped_stats},
] + [
    {"distribution": f"top{k}", **topk_stats[str(k)]}
    for k in TOP_K
]
with summary_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(summary_rows[0])
    )
    writer.writeheader()
    writer.writerows(summary_rows)

write_json(AUDIT / "xlkd01_cell06_gate.json", {
    "schema_version": "xlkd01_cell06_v1",
    "created_at_utc": now(),
    "status": "passed",
    "mapped_mass_gate": mapped_gate,
    "finite_logit_gate": finite_gate,
    "position_mask_gate": mask_gate,
    "top_k_gate": topk_gate,
    "overall_gate": overall_gate,
    "selected_top_k": selected_top_k,
    "decision": decision,
    "full_training_authorized": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01 CELL 6 — PROBABILITY-MASS GATE")
print("=" * 78)
print(
    "Mapped mass mean / p05   :",
    f'{mapped_stats["mean"]:.6f} / {mapped_stats["p05"]:.6f}',
)
for k in TOP_K:
    print(
        f"Top-{k:<3} mean / p05         :",
        f'{topk_stats[str(k)]["mean"]:.6f} / '
        f'{topk_stats[str(k)]["p05"]:.6f}',
    )
print("Mapped / top-k gate       :", mapped_gate, "/", topk_gate)
print("Selected top-k            :", selected_top_k)
print("Decision                  :", decision.upper())
print("Full training             : NOT AUTHORIZED")
print("Progress                  : 6 / 7 code cells")


XLKD-01 CELL 6 — PROBABILITY-MASS GATE
Mapped mass mean / p05   : 0.996712 / 0.995073
Top-32  mean / p05         : 0.208376 / 0.061839
Top-64  mean / p05         : 0.244816 / 0.081988
Top-128 mean / p05         : 0.296975 / 0.110990
Mapped / top-k gate       : True / False
Selected top-k            : None
Decision                  : STOP_SPARSE_TOPK_BRANCH
Full training             : NOT AUTHORIZED
Progress                  : 6 / 7 code cells


## Cell 7 — Package và authorize next stage hoặc stop branch


In [7]:
STARTED = time.perf_counter()
if overall_gate:
    NEXT_MARKER = OUT / "XLKD02_MAPPED_LOGIT_LOSS_PREFLIGHT_AUTHORIZED.json"
    write_json(NEXT_MARKER, {
        "schema_version": "xlkd02_mapped_logit_loss_preflight_authorized_v1",
        "created_at_utc": now(),
        "status": "authorized",
        "required_next": (
            "XLKD-02 fixed-batch mapped Logit-KD loss, backward, "
            "gradient-ratio and one-batch overfit preflight"
        ),
        "temperature": TEMPERATURE,
        "alpha_kd": 0.5,
        "selected_top_k": selected_top_k,
        "probability_mass_summary": {
            "relative_path": SUMMARY_PATH.relative_to(OUT).as_posix(),
            "sha256": sha256(SUMMARY_PATH),
        },
        "full_training_authorized": False,
        "validation_selection_authorized": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    })
else:
    NEXT_MARKER = OUT / "XLKD_SPARSE_TOPK_BRANCH_STOPPED.json"
    write_json(NEXT_MARKER, {
        "schema_version": "xlkd_sparse_topk_branch_stopped_v1",
        "created_at_utc": now(),
        "status": "stopped",
        "reason": (
            "No preregistered top-k candidate retained the required softened "
            "Teacher probability mass at T=4."
            if not topk_gate else
            "One or more mapped-mass, finite-logit or position-mask gates failed."
        ),
        "mapped_mass_gate": mapped_gate,
        "top_k_gate": topk_gate,
        "selected_top_k": selected_top_k,
        "allowed_next_actions": [
            "Report the negative result and retain Sequence KD R1.",
            "Register a new protocol for dense/on-the-fly mapped Logit KD.",
            "Do not loosen thresholds post hoc in this branch.",
        ],
        "full_training_authorized": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    })

tracked = (
    sorted(AUDIT.glob("*.json"))
    + sorted(CONFIG.glob("*.json"))
    + sorted(RESULT.glob("*"))
)
manifest_rows = []
for path in tracked:
    manifest_rows.append({
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
manifest_path = MANIFEST / "xlkd01_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
sums_path = MANIFEST / "SHA256SUMS"
sums_path.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

marker = {
    "schema_version": "xlkd01_probability_mass_preflight_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "method": "Cross-Vocabulary Mapped Logit KD",
    "sample_size": 1024,
    "temperature": TEMPERATURE,
    "mapped_mass_gate": mapped_gate,
    "top_k_gate": topk_gate,
    "overall_gate": overall_gate,
    "selected_top_k": selected_top_k,
    "decision": decision,
    "next_marker": {
        "relative_path": NEXT_MARKER.relative_to(OUT).as_posix(),
        "sha256": sha256(NEXT_MARKER),
    },
    "full_training_authorized": False,
    "manifest": {
        "csv": {
            "relative_path": manifest_path.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_path),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": sums_path.relative_to(OUT).as_posix(),
            "sha256": sha256(sums_path),
        },
    },
    "training_started": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
MARKER_PATH = OUT / "XLKD01_PROBABILITY_MASS_PREFLIGHT_COMPLETE.json"
write_json(MARKER_PATH, marker)

print("=" * 78)
print("XLKD-01 FINAL STATUS")
print("=" * 78)
print("Status                    : COMPLETE")
print("Mapped mass gate          :", "PASSED" if mapped_gate else "FAILED")
print("Sparse top-k gate         :", "PASSED" if topk_gate else "FAILED")
print("Selected top-k            :", selected_top_k)
print("Decision                  :", decision.upper())
print("Full training             : NOT AUTHORIZED")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 7 / 7 code cells")


XLKD-01 FINAL STATUS
Status                    : COMPLETE
Mapped mass gate          : PASSED
Sparse top-k gate         : FAILED
Selected top-k            : None
Decision                  : STOP_SPARSE_TOPK_BRANCH
Full training             : NOT AUTHORIZED
Validation / Test         : NOT READ / SEALED
Progress                  : 7 / 7 code cells
